# 🔧 DEX Extractor — APK Builder

Builds the Android APK directly from GitHub.

## متطلبات التشغيل
- ✅ Google account (لفتح Colab)
- ✅ اتصال بالإنترنت
- ✅ لا يحتاج أي تثبيت على جهازك

## خطوات الاستخدام
1. اضغط **Runtime → Run all** أو شغّل كل cell بالترتيب
2. انتظر حوالي 10-15 دقيقة
3. الـ APK ينزل تلقائياً في آخر cell

---

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 1 — Install Java 17
# ════════════════════════════════════════════════════════════════
print('📦 Installing Java 17...')
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

!java -version
print('✅ Java 17 ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 2 — Download project from GitHub
# ════════════════════════════════════════════════════════════════
import urllib.request, zipfile, os, shutil

GITHUB_URL = 'https://github.com/DanyFox12/New-work/raw/main/dex-extractor-v2.zip'
ZIP_PATH   = '/content/dex-extractor-v2.zip'
PROJ_DIR   = '/content/dex-extractor'

# Clean previous build if any
if os.path.exists(PROJ_DIR):
    shutil.rmtree(PROJ_DIR)

print(f'📥 Downloading from GitHub...')
urllib.request.urlretrieve(GITHUB_URL, ZIP_PATH)
print(f'   Size: {os.path.getsize(ZIP_PATH)/1024:.1f} KB')

print('📂 Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(PROJ_DIR)

os.chdir(PROJ_DIR)
!ls -la
print('✅ Project ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 3 — Install Android SDK + NDK + CMake
# ════════════════════════════════════════════════════════════════
import os

SDK_ROOT = '/content/android-sdk'
os.makedirs(f'{SDK_ROOT}/cmdline-tools', exist_ok=True)

print('📦 Downloading Android command-line tools...')
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip \
    -O /tmp/cmdtools.zip
!unzip -q /tmp/cmdtools.zip -d {SDK_ROOT}/cmdline-tools
!mv {SDK_ROOT}/cmdline-tools/cmdline-tools {SDK_ROOT}/cmdline-tools/latest

# Set SDK environment variables
os.environ['ANDROID_HOME']     = SDK_ROOT
os.environ['ANDROID_SDK_ROOT'] = SDK_ROOT
os.environ['PATH'] = (
    f'{SDK_ROOT}/cmdline-tools/latest/bin:'
    f'{SDK_ROOT}/platform-tools:'
    f'{SDK_ROOT}/build-tools/34.0.0:'
    + os.environ['PATH']
)

print('📝 Accepting licenses...')
!yes | sdkmanager --licenses > /dev/null 2>&1

print('📦 Installing SDK components (this takes ~5 min)...')
!sdkmanager --install \
    "platforms;android-34" \
    "build-tools;34.0.0" \
    "ndk;26.1.10909125" \
    "cmake;3.22.1" \
    2>&1 | tail -5

print('✅ Android SDK + NDK + CMake installed')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 4 — Configure local.properties
# ════════════════════════════════════════════════════════════════
import os, glob

os.chdir('/content/dex-extractor')

SDK_ROOT = '/content/android-sdk'

# Find NDK path dynamically
ndk_paths = glob.glob(f'{SDK_ROOT}/ndk/*')
NDK_PATH  = ndk_paths[0] if ndk_paths else f'{SDK_ROOT}/ndk/26.1.10909125'

with open('local.properties', 'w') as f:
    f.write(f'sdk.dir={SDK_ROOT}\n')
    f.write(f'ndk.dir={NDK_PATH}\n')

print(f'local.properties written:')
!cat local.properties
print('✅ Done')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 5 — Download Gradle 8.2
# ════════════════════════════════════════════════════════════════
import os

GRADLE_DIR = '/content/gradle'

print('📦 Downloading Gradle 8.2...')
!wget -q https://services.gradle.org/distributions/gradle-8.2-bin.zip \
    -O /tmp/gradle.zip
os.makedirs(GRADLE_DIR, exist_ok=True)
!unzip -q /tmp/gradle.zip -d {GRADLE_DIR}

GRADLE_BIN = f'{GRADLE_DIR}/gradle-8.2/bin/gradle'
!chmod +x {GRADLE_BIN}
!{GRADLE_BIN} --version

print('✅ Gradle 8.2 ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 6 — BUILD APK
# ════════════════════════════════════════════════════════════════
import os

os.chdir('/content/dex-extractor')

SDK_ROOT   = '/content/android-sdk'
GRADLE_BIN = '/content/gradle/gradle-8.2/bin/gradle'

os.environ['ANDROID_HOME']     = SDK_ROOT
os.environ['ANDROID_SDK_ROOT'] = SDK_ROOT
os.environ['JAVA_HOME']        = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = (
    f'{SDK_ROOT}/cmdline-tools/latest/bin:'
    f'{SDK_ROOT}/build-tools/34.0.0:'
    f'{SDK_ROOT}/platform-tools:'
    '/usr/lib/jvm/java-17-openjdk-amd64/bin:'
    + os.environ['PATH']
)

print('🔨 Building APK (debug)...')
print('   This may take 5-10 minutes on first build\n')

ret = os.system(f'{GRADLE_BIN} assembleDebug --no-daemon --stacktrace 2>&1')

if ret == 0:
    print('\n✅ BUILD SUCCESSFUL')
else:
    print('\n❌ BUILD FAILED — check output above')

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 7 — Download APK
# ════════════════════════════════════════════════════════════════
import glob, os
from google.colab import files

apks = glob.glob(
    '/content/dex-extractor/app/build/outputs/apk/**/*.apk',
    recursive=True
)

if apks:
    apk = apks[0]
    size_mb = os.path.getsize(apk) / (1024 * 1024)
    print(f'\n🎉 APK ready!')
    print(f'   Path : {apk}')
    print(f'   Size : {size_mb:.1f} MB')
    print('\n📥 Downloading...')
    files.download(apk)
    print('✅ Done! Install on your rooted device.')
else:
    print('❌ APK not found')
    print('   Run Step 6 again and check for errors')